# AutoDrive 2.0

## I. Getting Started Guide

### 1. AutoDrive Model Introduction

AutoDrive is an end-to-end temporal model in VisionPilot for estimating **distance-to-CIPO**, **road curvature**, and **CIPO presence probability** from two consecutive front-camera frames.

The network follows a compact **Backbone => Temporal Fusion Head => Multi-task Regression/Classification Heads** design:

- **Shared Backbone**: extracts high-level features from `image_prev` and `image_curr` using the same architecture family as AutoSpeed (1024×512 input, 2:1 aspect ratio).
- **Temporal Fusion**: concatenates/combines final-stage features from both frames to encode short-term motion cues.
- **Prediction Heads**: output three values per frame pair: normalized distance (`d_norm`), normalized curvature, and CIPO logit.

At inference time, common post-processing is:

- `distance_m = 150 × (1 - clamp(d_norm, 0, 1))`
- `curvature_1_per_m = curvature_pred × CURV_SCALE`
- `cipo_prob = sigmoid(flag_logit)` (often thresholded at 0.65)

For best results, match training preprocessing exactly (50° center crop + resize to 1024×512 + ImageNet normalization).

### 2. Environment Setup

**If you already installed this repo’s Python dependencies elsewhere, skip to section `3`.**

Clone this repository (set `REPO_URL` / `REPO_DIR` to match your fork or mirror).

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/autowarefoundation/autoware.privately-owned-vehicles.git"
REPO_DIR = os.path.abspath("./autoware.privately-owned-vehicles")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=False)

Install from **`Models/requirements.txt`** (adjust the path if your clone directory name differs).

In [ ]:
%pip install --upgrade pip
%pip install -r autoware.privately-owned-vehicles/Models/requirements.txt

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download model weight files using the links below. The rest of this tutorial assumes they are placed in `./weights/`.

**AutoDrive model weights — 2:1 aspect ratio, 1024px by 512px input image**
- [Link to Download Pytorch Model Weights *.pt](<https://drive.google.com/drive/u/1/folders/182h_9eBHroMCOfQHJiXVgrNq7zHx7Qws?dmr=1&ec=wgc-drive-hero-goto>)

You can also use the next cell with `gdown` once your Google Drive file ID is ready.

In [ ]:
%pip install gdown

import os
import gdown

base_dir = os.getcwd()
model_dir = "./weights"
os.makedirs(model_dir, exist_ok=True)

metadata = {
    "pt": {
        "url": "https://drive.google.com/uc?id=<ADD_FILE_ID_HERE>",
        "filepath": os.path.join(base_dir, model_dir, "AutoDrive.pt"),
    }
}

for weight_type, info in metadata.items():
    url = info["url"]
    filepath = info["filepath"]

    if os.path.exists(filepath):
        print(f"AutoDrive {weight_type} weights already exist at {filepath}. Skipping download.")
    else:
        if "<ADD_FILE_ID_HERE>" in url:
            print("Please replace <ADD_FILE_ID_HERE> with your Google Drive file ID first.")
        else:
            print(f"Downloading AutoDrive {weight_type} weights...")
            gdown.download(url, filepath, quiet=False)

## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup** first.

### 1. Single Pair Inference

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import torch
import torchvision.transforms.functional as TF
from PIL import Image

# --- Repository root ---
ROOT = Path(".").resolve()
for _ in range(8):
    if (ROOT / "Models" / "training" / "train_auto_drive.py").is_file():
        break
    ROOT = ROOT.parent
else:
    raise RuntimeError("Could not find repo root; set ROOT = Path('/your/checkout')")
sys.path.insert(0, str(ROOT))

from Models.data_parsing.AutoDrive.zod.zod_utils import get_images_blur_dir
from Models.data_utils.load_data_auto_drive import CURV_SCALE, _center_crop_50deg_resize, _read_hfov_deg
from Models.model_components.autodrive.autodrive_network import AutoDrive

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD = [0.229, 0.224, 0.225]
_D_MAX = 150.0
_WHEELBASE_M = 2.984
_STEER_RATIO = 16.8
_CIPO_THRESH = 0.65


def curvature_to_steer_deg(c_1_per_m: float) -> float:
    return math.degrees(math.atan(c_1_per_m * _WHEELBASE_M) * _STEER_RATIO)


def to_model_tensor(np_rgb_hwc_uint8, device):
    t = TF.to_tensor(np_rgb_hwc_uint8)
    t = TF.normalize(t, _IMAGENET_MEAN, _IMAGENET_STD)
    return t.unsqueeze(0).to(device)


def resolve_checkpoint() -> Path:
    cands = [
        Path(os.environ.get("AUTODRIVE_CHECKPOINT", "")).expanduser(),
        Path("./weights/AutoDrive.pt"),
        Path("./weights/AutoDrive_best.pth"),
        Path("./weights/AutoDrive_last.pth"),
    ]
    for p in cands:
        if str(p) and p.is_file():
            return p.resolve()
    raise FileNotFoundError(
        "Set AUTODRIVE_CHECKPOINT or place AutoDrive*.pt under ./weights/"
    )


# ============ edit these ============
DATASET_ROOT = Path.home() / "Downloads" / "data" / "zod"  # dataset root (ZOD-style)
SEQUENCE_ID = "001451"  # six digits or integer string
# ====================================

seq_id = f"{int(SEQUENCE_ID):06d}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = resolve_checkpoint()
print("Checkpoint:", ckpt_path)

model = AutoDrive().to(device)
ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
model.load_state_dict(sd, strict=False)
model.eval()

label_dir = DATASET_ROOT / "labels" / seq_id
if not label_dir.is_dir():
    raise FileNotFoundError(f"Labels not found: {label_dir}")

jsons = sorted(label_dir.glob("*.json"))
hfov = _read_hfov_deg(DATASET_ROOT, seq_id)
img_dir = get_images_blur_dir(DATASET_ROOT, seq_id)

paths = []
for lf in jsons[:2]:
    with open(lf) as fh:
        rec = json.load(fh)
    paths.append(img_dir / rec["image"])

prev_np = _center_crop_50deg_resize(Image.open(paths[0]).convert("RGB"), hfov)
curr_np = _center_crop_50deg_resize(Image.open(paths[1]).convert("RGB"), hfov)
t_prev = to_model_tensor(prev_np, device)
t_curr = to_model_tensor(curr_np, device)

with torch.no_grad():
    d_norm, curv_norm, flag_logit = model(t_prev, t_curr)

d_norm_v = float(d_norm.squeeze().cpu())
curv_v = float(curv_norm.squeeze().cpu()) * CURV_SCALE
prob = float(torch.sigmoid(flag_logit.squeeze()).cpu())
dist_m = _D_MAX * (1.0 - max(0.0, min(1.0, d_norm_v)))
steer_deg = curvature_to_steer_deg(curv_v)
cipo_pred = 1 if prob >= _CIPO_THRESH else 0

print(f"distance (m):     {dist_m:.1f}")
print(f"curvature (1/m): {curv_v:.5f}")
print(f"steer proxy (°): {steer_deg:+.2f}")
print(f"CIPO prob:        {prob:.3f}  →  pred={cipo_pred} (thresh={_CIPO_THRESH})")

### 2. Sequence Inference

Run inference across a short sequence using consecutive frame pairs `(t-1, t)` and print predictions for quick debugging.

In [ ]:
MAX_PAIRS = 30

records = []
for lf in sorted((DATASET_ROOT / "labels" / seq_id).glob("*.json")):
    with open(lf) as fh:
        rec = json.load(fh)
    img_p = img_dir / rec["image"]
    if img_p.is_file():
        records.append((str(img_p), rec))

if len(records) < 2:
    raise RuntimeError("Need at least two frames")

hfov = _read_hfov_deg(DATASET_ROOT, seq_id)
n = min(MAX_PAIRS, len(records) - 1)

has_gt = "cipo_detected" in records[1][1]
if has_gt:
    print(f"{'i':>4}  {'dist_m':>8}  {'steer°':>8}  {'p(CIPO)':>8}  {'lbl dist':>10}  {'lbl CIPO':>8}")
else:
    print(f"{'i':>4}  {'dist_m':>8}  {'steer°':>8}  {'p(CIPO)':>8}")
print("-" * 62)

for i in range(1, 1 + n):
    prev_np = _center_crop_50deg_resize(Image.open(records[i - 1][0]).convert("RGB"), hfov)
    curr_np = _center_crop_50deg_resize(Image.open(records[i][0]).convert("RGB"), hfov)
    tp = to_model_tensor(prev_np, device)
    tc = to_model_tensor(curr_np, device)
    with torch.no_grad():
        d_n, c_n, fl = model(tp, tc)
    dnv = float(d_n.squeeze().cpu())
    cv = float(c_n.squeeze().cpu()) * CURV_SCALE
    pr = float(torch.sigmoid(fl.squeeze()).cpu())
    dm = _D_MAX * (1.0 - max(0.0, min(1.0, dnv)))
    sdg = curvature_to_steer_deg(cv)
    if has_gt:
        lbl = records[i][1]
        gf = 1 if lbl.get("cipo_detected") else 0
        raw = lbl.get("distance_to_in_path_object")
        gd = float(raw) if (raw is not None and gf) else None
        gds = f"{gd:8.1f}" if gd is not None else "       —"
        print(f"{i:4d}  {dm:8.1f}  {sdg:+8.2f}  {pr:8.3f}  {gds}  {gf:8d}")
    else:
        print(f"{i:4d}  {dm:8.1f}  {sdg:+8.2f}  {pr:8.3f}")

## III. Model Training

AutoDrive training is launched from `Models/training/train_auto_drive.py`.

Typical workflow:

- Prepare dataset root with labels/images in expected format.
- Optionally initialize backbone from `autospeed.pt` using `--autospeed-ckpt`.
- Train either `curvature` mode (head-focused) or `joint` mode (full network).
- Save checkpoints under `{root}/training/autodrive/<run-name>/checkpoints/`.

Example command:

```bash
python Models/training/train_auto_drive.py \
  --root /path/to/dataset_root \
  --run-name run_autodrive_experiment \
  --train-mode joint \
  --autospeed-ckpt /path/to/autospeed.pt \
  --epochs 50 \
  --batch-size 16 \
  --workers 4
```

For all training flags:

```bash
python Models/training/train_auto_drive.py --help
```